# Linear Probing
Linear probing of embeddings generated by Gigapath

## Requirements

In [2]:
import torch
import os 
import deeplake
from torch.utils.data import DataLoader
import torch.nn.functional as F
from sklearn.metrics import accuracy_score
from typing import (
    Any,
    Dict, 
    List,
    Tuple,
    Iterable,
    Optional,
    Callable
)
from tqdm import tqdm
import os
from math import inf

import torch
import deeplake
import torch.nn as nn
from torch.utils.data import DataLoader
from torch.utils.tensorboard import SummaryWriter
from torch.optim.lr_scheduler import CosineAnnealingLR

import pyvips

from requirements_emb import (
    get_args,
    set_seed,
    log_device,
    log_metrics,
    embedding_transform_fn,
    Network,
    NetworkHandler,
    CONFIG_DIR,
    BORDER_WIDTH,
    BASE_MODEL_DIR
)






/home/leolr-int/micromamba/envs/py312-poetry/lib/python3.12/site-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


## Training

In [ ]:
'''download = False
if download:

    from huggingface_hub import login
    login(token="-")
    
    encoder_dir = os.path.join(BASE_MODEL_DIR, "pre_trained_weights")
    encoder = Network("gigapath", encoder_dir).encoder'''

In [ ]:

def main():
    log_device()
    num_workers = max(1, (os.cpu_count() // 4))

    arg_path = os.path.join(CONFIG_DIR, "linear_probe.yaml")
    args = get_args(arg_path)
    set_seed(args["seed"])

    # Fixed dataset path
    ds_path = "/home/leolr-int/nfs/transformed_data/new_embeddings/Akoya/mixed_precision/dim_256/Train/gigapath"
    encoder_dir = os.path.join(BASE_MODEL_DIR, "pre_trained_weights")

    # Simplified logging directory
    log_dir = f"/home/leolr-int/nfs/transformed_data/weights/linear_probe_logs/experiment_{args.get('experiment_num', 1)}"
    writer = SummaryWriter(log_dir)
    
    # Fixed weights directory
    model_dir = "/home/leolr-int/nfs/transformed_data/weights"
    os.makedirs(model_dir, exist_ok=True)

    # Load the full dataset
    ds = deeplake.open_read_only(ds_path)
    
    # Use entire dataset as training data
    train_dataset = ds.pytorch(transform=embedding_transform_fn)

    train_loader = DataLoader(
        train_dataset,
        batch_size=args["batch_size"],
        shuffle=True,
        pin_memory=True,
        persistent_workers=True,
        num_workers=num_workers
    )

    # Initialize model - using 'gigapath' as encoder name
    model = Network("gigapath", encoder_dir, args["num_classes"], freeze_encoder=True)
    trainable_params = list(filter(lambda p: p.requires_grad, model.parameters()))

    criterion = nn.CrossEntropyLoss()
    optimizer = torch.optim.AdamW(trainable_params, lr=args["learning_rate"], weight_decay=args["weight_decay"])
    scheduler = CosineAnnealingLR(optimizer, args["epochs"], eta_min=args["eta_min"])

    network_handler = NetworkHandler(
        model=model,
        criterion=criterion,
        optimizer=optimizer,
        precision=args.get("precision", "mixed"),  # Default to mixed precision
        embedding_mode=True
    )

    min_train_loss = inf
    max_train_accuracy = -inf

    for epoch in range(1, args["epochs"] + 1):
        print("\n" + f"Epoch [{epoch}/{args['epochs']}]".center(BORDER_WIDTH))
        print(f"{'=' * BORDER_WIDTH}\n")

        writer.add_scalar("Learning Rate", scheduler.optimizer.param_groups[0]["lr"], epoch)

        train_loss, train_accuracy = network_handler.train_epoch(train_loader)
        log_metrics(writer=writer, loss=train_loss, prefix="Train", epoch=epoch, performance=train_accuracy)

        # Save best models based on training metrics
        if train_loss < min_train_loss:
            torch.save(model.fc.state_dict(), os.path.join(model_dir, "lowest_train_loss.pth"))
            min_train_loss = train_loss
            print("New minimum training loss — model saved.")
        
        if train_accuracy > max_train_accuracy:
            torch.save(model.fc.state_dict(), os.path.join(model_dir, "highest_train_accuracy.pth"))
            max_train_accuracy = train_accuracy
            print("New maximum training accuracy — model saved.")
            
    
        scheduler.step()

        print(f"{'=' * BORDER_WIDTH}\n")

    # Save final model
    torch.save(model.fc.state_dict(), os.path.join(model_dir, "final_model.pth"))
    print("Final model saved.")

    print("Run Summary:")
    print(f"Min Training Loss: {min_train_loss:.4f} | Max Training Accuracy: {max_train_accuracy:.4f}\n")

        
'''if __name__ == "__main__":
    main()'''


                               Devices                                
----------------------------------------------------------------------

                  GPU 0: NVIDIA GeForce RTX 3080 Ti                   
                       Memory Used : 920.44 MB                        
                      Memory Total: 12288.00 MB                       

----------------------------------------------------------------------
                     PyTorch CUDA Available: True                     


                             Epoch [1/50]                             



Training in progress: 100%|██████████| 710/710 [00:37<00:00, 18.98it/s, step_loss=0.306]



Train Statistics:
Loss: 0.4603 | Balanced Accuracy: 0.6737

New minimum training loss — model saved.
New maximum training accuracy — model saved.


                             Epoch [2/50]                             



Training in progress: 100%|██████████| 710/710 [00:36<00:00, 19.67it/s, step_loss=0.24] 



Train Statistics:
Loss: 0.3042 | Balanced Accuracy: 0.7814

New minimum training loss — model saved.
New maximum training accuracy — model saved.


                             Epoch [3/50]                             



...


Train Statistics:
Loss: 0.1979 | Balanced Accuracy: 0.8836

New minimum training loss — model saved.


                            Epoch [49/50]                             



Training in progress: 100%|██████████| 710/710 [00:40<00:00, 17.32it/s, step_loss=0.119]



Train Statistics:
Loss: 0.1979 | Balanced Accuracy: 0.8836



                            Epoch [50/50]                             



Training in progress: 100%|██████████| 710/710 [00:37<00:00, 18.74it/s, step_loss=0.285]



Train Statistics:
Loss: 0.1981 | Balanced Accuracy: 0.8836


Final model saved.
Run Summary:
Min Training Loss: 0.1979 | Max Training Accuracy: 0.8857



## Accuracy

In [4]:
def linear_probing(weights_path, dataset):
    
    state_dict = torch.load(weights_path)

    weight = state_dict['head.weight'] # shape: [5, 1536]
    bias = state_dict.get('head.bias', None)  # shape: [5]
   
    EMBEDDING_DIR = f"/home/leolr-int/nfs/transformed_data/new_embeddings/{dataset}"  # or KFBio, etc.
    embedding_path = os.path.join(EMBEDDING_DIR, "mixed_precision", "dim_256", "Train", "gigapath")

    ds = deeplake.open_read_only(embedding_path)
    ds_torch = ds.pytorch(transform=embedding_transform_fn)
    ds_loader = DataLoader(ds_torch, batch_size=64, shuffle=False)  # larger batch_size for efficiency
    weight = weight.cpu()
    bias = bias.cpu()

    all_preds = []
    all_labels = []

    for batch in tqdm(ds_loader, desc="Linear probing"):
        embedding, label, _ = batch  
        
        # Linear probing: logits = embedding @ W^T + b
        logits = embedding @ weight.T + bias

        preds = torch.argmax(F.softmax(logits, dim=1), dim=1)

        all_preds.append(preds)
        all_labels.append(label)

    # Concatenate all predictions and labels
    all_preds = torch.cat(all_preds)
    all_labels = torch.cat(all_labels)

    acc = accuracy_score(all_labels.cpu(), all_preds.cpu())

    print(f"Accuracy for {dataset} using Gigapath and linear probing on Train data: {acc:.4f}")



In [11]:
# test on Eric's weight
weights_path = "/home/leolr-int/AGGCPerturbations/model_weights/linear_probe_weights/single_precision/overlap_0.0/dim_256/gigapath/split_1/experiment_1/highest_balanced_accuracy.pth"
linear_probing(weights_path, 'Akoya')
linear_probing(weights_path, 'KFBio')
linear_probing(weights_path, 'KFBio_FDA5')

Linear probing: 100%|██████████| 2837/2837 [01:21<00:00, 34.71it/s]


Accuracy for Akoya using Gigapath and linear probing on Train data: 0.8778


Linear probing: 100%|██████████| 3086/3086 [01:30<00:00, 34.18it/s]


Accuracy for KFBio using Gigapath and linear probing on Train data: 0.8715


Linear probing: 100%|██████████| 3086/3086 [01:28<00:00, 35.03it/s]

Accuracy for KFBio_FDA5 using Gigapath and linear probing on Train data: 0.7808


In [6]:
# test on my weights (trained only using Subset3_Train_{i}_Akoya)
model_dir = "/home/leolr-int/nfs/transformed_data/weights"
my_weights = os.path.join(model_dir, "final_model.pth")

linear_probing(my_weights, 'Akoya')
linear_probing(my_weights, 'KFBio')
linear_probing(my_weights, 'KFBio_FDA5')
linear_probing(my_weights, 'KFBio_FDA1')

Linear probing: 100%|██████████| 2837/2837 [01:16<00:00, 37.08it/s]


Accuracy for Akoya using Gigapath and linear probing on Train data: 0.9292


Linear probing: 100%|██████████| 3086/3086 [01:23<00:00, 37.01it/s]


Accuracy for KFBio using Gigapath and linear probing on Train data: 0.8394


Linear probing: 100%|██████████| 3086/3086 [01:23<00:00, 37.07it/s]


Accuracy for KFBio_FDA5 using Gigapath and linear probing on Train data: 0.7740


Linear probing: 100%|██████████| 3086/3086 [01:23<00:00, 37.03it/s]

Accuracy for KFBio_FDA1 using Gigapath and linear probing on Train data: 0.8139
